# 04 Strategy Simulation

This notebook uses lap time assumptions and strategy simulation logic to compare Formula 1 race strategy options.

The focus is on a Mercedes-style strategy decision: comparing one-stop and two-stop race strategies, estimating total race time, and identifying optimal pit windows.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../visuals", exist_ok=True)

In [ ]:
df = pd.read_csv("../data/raw/synthetic_f1_race_data.csv")

df.head()

## Simulation Assumptions

The simulator estimates lap time using simplified race strategy assumptions.

It includes:

- Team baseline pace
- Driver adjustment
- Tyre compound effect
- Tyre degradation
- Fuel load effect
- Track evolution
- Pit stop time loss

The objective is to compare total race time across different strategy plans.

In [ ]:
n_laps = 58
pit_loss_seconds = 22.0

base_pace = {
    "Mercedes": 89.0
}

driver_adjustment = {
    "HAM": -0.10,
    "RUS": 0.05
}

compound_effect = {
    "Soft": -0.60,
    "Medium": 0.00,
    "Hard": 0.45
}

degradation_rate = {
    "Soft": 0.095,
    "Medium": 0.055,
    "Hard": 0.035
}

In [ ]:
def estimate_lap_time(driver, lap, compound, tyre_age):
    team = "Mercedes"

    fuel_effect = (n_laps - lap) * 0.035
    track_evolution = -lap * 0.008
    tyre_degradation = degradation_rate[compound] * tyre_age

    lap_time = (
        base_pace[team]
        + driver_adjustment[driver]
        + compound_effect[compound]
        + tyre_degradation
        + fuel_effect
        + track_evolution
    )

    return lap_time

In [ ]:
def simulate_strategy(driver, strategy_name, stints):
    rows = []
    total_time = 0

    for stint_number, (start_lap, end_lap, compound) in enumerate(stints, start=1):
        for lap in range(start_lap, end_lap + 1):
            tyre_age = lap - start_lap + 1
            lap_time = estimate_lap_time(driver, lap, compound, tyre_age)

            pit_stop = 1 if lap == start_lap and lap != 1 else 0

            if pit_stop:
                lap_time += pit_loss_seconds

            total_time += lap_time

            rows.append({
                "driver": driver,
                "strategy": strategy_name,
                "lap": lap,
                "stint": stint_number,
                "compound": compound,
                "tyre_age": tyre_age,
                "pit_stop": pit_stop,
                "lap_time_seconds": round(lap_time, 3),
                "cumulative_race_time": round(total_time, 3)
            })

    return pd.DataFrame(rows)

## Compare Strategy Options

Three Mercedes strategy options are simulated:

1. One-stop: Medium to Hard
2. Two-stop balanced: Medium to Hard to Medium
3. Two-stop aggressive: Soft to Medium to Hard

In [ ]:
strategies = {
    "One-stop M-H": [(1, 24, "Medium"), (25, 58, "Hard")],
    "Two-stop M-H-M": [(1, 18, "Medium"), (19, 38, "Hard"), (39, 58, "Medium")],
    "Two-stop S-M-H": [(1, 15, "Soft"), (16, 36, "Medium"), (37, 58, "Hard")]
}

simulation_results = []

for strategy_name, stints in strategies.items():
    strategy_df = simulate_strategy("HAM", strategy_name, stints)
    simulation_results.append(strategy_df)

simulation_df = pd.concat(simulation_results, ignore_index=True)

simulation_df.head()

In [ ]:
strategy_summary = (
    simulation_df.groupby("strategy")
    .agg(
        total_race_time_seconds=("lap_time_seconds", "sum"),
        pit_stops=("pit_stop", "sum"),
        average_lap_time=("lap_time_seconds", "mean")
    )
    .sort_values("total_race_time_seconds")
)

strategy_summary

In [ ]:
strategy_summary["time_gap_to_best"] = (
    strategy_summary["total_race_time_seconds"] 
    - strategy_summary["total_race_time_seconds"].min()
)

strategy_summary

In [ ]:
strategy_summary["total_race_time_seconds"].sort_values().plot(kind="bar")

plt.title("Total Race Time by Strategy")
plt.xlabel("Strategy")
plt.ylabel("Total Race Time (seconds)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../visuals/total_race_time_by_strategy.png", bbox_inches="tight")
plt.show()

## Cumulative Race Time Comparison

This chart shows how strategy performance evolves across the race.

A strategy can be slower early but recover later depending on tyre degradation and pit stop timing.

In [ ]:
plt.figure(figsize=(11, 6))

for strategy in simulation_df["strategy"].unique():
    strategy_data = simulation_df[simulation_df["strategy"] == strategy]
    plt.plot(
        strategy_data["lap"],
        strategy_data["cumulative_race_time"],
        label=strategy
    )

plt.title("Cumulative Race Time by Strategy")
plt.xlabel("Lap")
plt.ylabel("Cumulative Race Time (seconds)")
plt.legend()
plt.tight_layout()
plt.savefig("../visuals/cumulative_race_time_by_strategy.png", bbox_inches="tight")
plt.show()

## Pit Window Simulation

This section tests different one-stop pit windows for a Medium to Hard strategy.

The goal is to find which pit lap produces the lowest total race time.

In [ ]:
pit_window_results = []

for pit_lap in range(16, 33):
    stints = [(1, pit_lap - 1, "Medium"), (pit_lap, 58, "Hard")]
    strategy_name = f"Pit lap {pit_lap}"

    strategy_df = simulate_strategy("HAM", strategy_name, stints)
    total_time = strategy_df["lap_time_seconds"].sum()

    pit_window_results.append({
        "pit_lap": pit_lap,
        "total_race_time_seconds": total_time
    })

pit_window_df = pd.DataFrame(pit_window_results)
pit_window_df["time_gap_to_best"] = (
    pit_window_df["total_race_time_seconds"]
    - pit_window_df["total_race_time_seconds"].min()
)

pit_window_df.sort_values("total_race_time_seconds").head()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    pit_window_df["pit_lap"],
    pit_window_df["time_gap_to_best"],
    marker="o"
)

plt.title("One-stop Pit Window Simulation")
plt.xlabel("Pit Lap")
plt.ylabel("Time Gap to Best Strategy (seconds)")
plt.tight_layout()
plt.savefig("../visuals/pit_window_simulation.png", bbox_inches="tight")
plt.show()

## Strategy Recommendation

The recommended strategy is selected based on lowest simulated total race time.

In [ ]:
best_strategy = strategy_summary.sort_values("total_race_time_seconds").iloc[0]

print("Recommended strategy:")
print(best_strategy.name)
print()
print(f"Total race time: {best_strategy['total_race_time_seconds']:.2f} seconds")
print(f"Pit stops: {int(best_strategy['pit_stops'])}")
print(f"Average lap time: {best_strategy['average_lap_time']:.2f} seconds")

In [ ]:
best_pit_lap = pit_window_df.sort_values("total_race_time_seconds").iloc[0]

print(f"Optimal one-stop pit lap: Lap {int(best_pit_lap['pit_lap'])}")
print(f"Time gap to best: {best_pit_lap['time_gap_to_best']:.2f} seconds")

## Save Strategy Outputs

The simulation outputs are saved for documentation and README reporting.

In [ ]:
simulation_df.to_csv("../data/processed/strategy_simulation_results.csv", index=False)
strategy_summary.to_csv("../data/processed/strategy_summary.csv")
pit_window_df.to_csv("../data/processed/pit_window_simulation.csv", index=False)

print("Strategy simulation outputs saved.")

## Key Findings

- Race strategy depends on the trade-off between tyre degradation and pit stop time loss.
- A one-stop strategy avoids an extra pit stop but may suffer from tyre degradation late in the race.
- A two-stop strategy gives fresher tyres but adds another pit stop time penalty.
- Pit window timing can change total race time by several seconds.
- The simulation framework can support strategy decisions such as undercut, overcut and alternative tyre plans.